---

# C4. Exercițiu individual: construirea unui mini-prompt de adnotare

În acest exercițiu construiești un prompt mic de adnotare pentru comentarii politice.
- Intelegem cum se construiește un prompt: rol, variabile, definiții, reguli și format JSON.
- Alegemdouă axe proprii sau două axe din curs și vei testa promptul pe 5 comentarii.


## Pasul 0 . Configurare

In [1]:
import os, json, re, random
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
# caută .env urcând din folderul curent

ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

# DeepSeek
deepseek_client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)
DEEPSEEK_MODEL = "deepseek-chat"
# Gemini prin OpenAI-compatible API
gemini_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

GEMINI_MODEL = "gemini-2.5-flash-lite"
# alegem modelul pentru demo

USE_GEMINI = True
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Root project:", ROOT)
print("DeepSeek key:", os.getenv("DEEPSEEK_API_KEY") is not None)
print("Gemini key:", os.getenv("GEMINI_API_KEY") is not None)
print("Model folosit:", model_now)
print("OK")

Root project: c:\Users\mihai\OneDrive\Desktop\Facultate\Master\Semestre\S4\Ai_engineering\echochamber-project-team-6
DeepSeek key: True
Gemini key: True
Model folosit: gemini-2.5-flash-lite
OK


## Corpus

In [2]:
import pandas as pd
import random

corpus = pd.read_json("../../data/cleaned/corpus_youtube_sample.jsonl", lines=True)

print(len(corpus), "comentarii")
print("Câmpuri:", list(corpus.columns))

for _, c in corpus.sample(3).iterrows():
    print(f"[{c['source_channel'][:30]}] {c['text'][:80]}")

420 comentarii
Câmpuri: ['id', 'source_channel', 'video_title', 'text']
[RecorderRomania] La pușcărie cu Savonea, Arsenie, Voineag și toți corupții din partide care au pu
[CălinGeorgescu-CanalulOficial] Pacea de la București este morandumul inițiat de către Diana Șosoaca. Văd că vă 
[StirileProTV] Sufletul Pământului în rezervorul şi motorul başinilor dracului.


### Pasul 1 Alege două axe

Alege două axe pe care vrei să le codezi.
Poți folosi axe din curs:
- institutional
- legitimare
- epistemic
- geopolitic
- mobilizare
Sau poți propune axe proprii:
- media_distrust
- elite_blame
- religious_frame
- fear
- irony
- people_vs_elite
- anti_corruption
- national_identity
Condiție: fiecare axă trebuie să aibă valori clare.
Pentru acest exercițiu folosim o scală simplă:
0 = absent
1 = prezent


In [3]:
# modifica dupa preferinte

AXA_1 = "anti_corruption"
AXA_2 = "fear"

## Pasul 2 — Definește axele
Scrie mai jos, în propriile cuvinte, ce înseamnă fiecare axă.
Exemplu:
media_distrust = comentariul exprimă neîncredere în presă, jurnaliști, televiziuni sau media mainstream.
religious_frame = comentariul folosește limbaj religios pentru a interpreta politica.

In [4]:
AXA_1_DEFINITION = """
anti_corruption măsoară dacă textul exprimă neîncredere în autorități, politicieni, instituții în general.
0 = absent
1 = prezent
2 = dominant
"""
AXA_2_DEFINITION = """
fear măsoară dacă textul exprimă teamă sau anxietate legate de o anumită situație sau grup.
0 = absent
1 = prezent
2 = dominant
"""

## Pasul 3 — Construiește mini-promptul
Promptul trebuie să conțină:
1. rolul modelului;
2. sarcina;
3. definițiile celor două axe;
4. regulile de codare;
5. formatul JSON.
Important:
- nu cere modelului să identifice direct „bula”;
- nu cere text liber;
- returnează doar JSON valid.

In [5]:
MINI_PROMPT = f"""
Ești un student la științe politice care adnotează comentarii online pentru un proiect.
SARCINĂ:
Adnotează comentariul folosind două axe:
1. {AXA_1}
2. {AXA_2}
CÂMPURI:
target = ținta politică principală din comentariu
stance = poziția față de target: pro / anti / neutru / ambiguu / none
tone = modul dominant de formulare: acuzator / ironic / mobilizator / defensiv / afectiv / neutru
{AXA_1} = 0 / 1 / 2
{AXA_2} = 0 / 1 / 2
DEFINIȚII:
{AXA_1_DEFINITION}
{AXA_2_DEFINITION}
REGULI:
1. Codează doar ce apare în comentariu, titlu sau canal.
2. Nu inventa informații externe.
3. Dacă nu există target politic, folosește target="none" și stance="none".
4. Dacă textul este ironic, codează sensul intenționat, nu sensul literal.
5. Pentru axe: 0 = absent, 1 = prezent, 2 = dominant.
6. Nu atribui direct o bulă discursivă.
7. Returnează doar JSON valid.
FORMAT OUTPUT:
{{
  "target": "",
  "stance": "",
  "tone": "",
  "{AXA_1}": 0,
  "{AXA_2}": 0
}}
"""
print(MINI_PROMPT)


Ești un student la științe politice care adnotează comentarii online pentru un proiect.
SARCINĂ:
Adnotează comentariul folosind două axe:
1. anti_corruption
2. fear
CÂMPURI:
target = ținta politică principală din comentariu
stance = poziția față de target: pro / anti / neutru / ambiguu / none
tone = modul dominant de formulare: acuzator / ironic / mobilizator / defensiv / afectiv / neutru
anti_corruption = 0 / 1 / 2
fear = 0 / 1 / 2
DEFINIȚII:

anti_corruption măsoară dacă textul exprimă neîncredere în autorități, politicieni, instituții în general.
0 = absent
1 = prezent
2 = dominant


fear măsoară dacă textul exprimă teamă sau anxietate legate de o anumită situație sau grup.
0 = absent
1 = prezent
2 = dominant

REGULI:
1. Codează doar ce apare în comentariu, titlu sau canal.
2. Nu inventa informații externe.
3. Dacă nu există target politic, folosește target="none" și stance="none".
4. Dacă textul este ironic, codează sensul intenționat, nu sensul literal.
5. Pentru axe: 0 = absent,

## Pasul 4 — Alege 5 comentarii de test
Folosim un eșantion mic. Nu adnotăm tot corpusul.
Schimbă `random_state` ca să primești alte comentarii.

In [6]:
TESTS = corpus.sample(5, random_state=15)
TESTS[["id", "source_channel", "video_title", "text"]].head()

,id,source_channel,video_title,text
190,yt_iH8jB4NlV9Y_UgyxzJOJN2dXFNHdQL94AaABAg,georgesimionoficial,Episodul 2: Cum ne-au furat alegerile - Turism...,Nu a aratat o dovoda clara doar isi continua n...
97,yt_Pk7Qkhnt4Bs_UgyeY3JjWWUu4UTy8uV4AaABAg,NicusorDanRO,🟢 LIVE - Declarații de presă susținute după pa...,"Nimeni nu se naște președinte . Nicușor Dan , ..."
73,yt_q6wy7cs5RuU_UgxtjWylQ8u3U0mA5u14AaABAg,RecorderRomania,Judecătoarea Raluca Moroșanu: „Dacă nu se schi...,Cat adevar cate dovezii ne prezentati si incon...
290,yt_R-wmsuFxku4_UgyZ1UUIKqIDPM-tch94AaABAg,@CălinGeorgescu-CanalulOficial,Călin Georgescu - Tema pentru Guvernul Românie...,"FAMILIA GEORGESCU, CRISTELA GEORGESCU, CALIN G..."
40,yt_yEuctxNb4O0_UgyJ9fwjn8yOBB6XWtt4AaABAg,NicusorDanRO,🟢 LIVE - Întâlnire la Palatul Cotroceni cu mag...,Baliverne. Justiția e pe bani indiferent ca vb...


## Pasul 5 — Rulează promptul pe cele 5 comentarii
Pentru fiecare comentariu:
1. trimitem canalul, titlul video și textul;
2. modelul returnează JSON;
3. citim rezultatul și verificăm dacă are sens.

In [7]:
USE_GEMINI = True
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Using:", model_now)

Using: gemini-2.5-flash-lite


In [8]:
def llm(system, user, max_tokens=700):
    response = client_now.chat.completions.create(
        model=model_now,
        temperature=0,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ]
    )
    return response.choices[0].message.content

In [9]:
results = []
for _, row in TESTS.iterrows():
    USER = f"""
CANAL:
{row.get("source_channel", "")}
TITLU VIDEO:
{row.get("video_title", "")}
COMENTARIU:
<<< {row["text"]} >>>
"""
    raw = llm(MINI_PROMPT, USER, max_tokens=300)
    print("=" * 80)
    print("COMENTARIU:")
    print(row["text"])
    print()
    print("OUTPUT MODEL:")
    print(raw)
    results.append({
        "id": row["id"],
        "text": row["text"],
        "model_output": raw
    })

COMENTARIU:
Nu a aratat o dovoda clara doar isi continua narativa. In video sunt doar niste oameni, habar nu am cine sunt (ca nu sunt prezentati), care isi dau cu parerea fara sa produca un act sau video cu ce spun. Iar ceea ce spun este de domeniul SF-ului. Nu mentineaza deloc ajutorul acordat de Romania Moldovei si programele dintre cele 2 tari. Apropo, daca Moldova intre in UE atunci nu o sa mai fie nevoie de granite, o sa fie un fel de Unire. Ce vad in partidul AUR doar oameni care fac orice pentru putere si sacrifica pe oricine pentru asi atincge scopul. Domnul Simion nu are maturitatea si onoarea sa inteleaga ca tara trece printr-o criza si trebuie sa isi puna interesele proprii pe al doilea loc si sa contribuie cu solutii la iesirea din criza. Atatea se poate.

OUTPUT MODEL:
```json
{
  "target": "AUR",
  "stance": "anti",
  "tone": "acuzator",
  "anti_corruption": 2,
  "fear": 0
}
```
COMENTARIU:
Nimeni nu se naște președinte . Nicușor Dan , cel mai bun președinte de la revoluț

## Pasul 6 — Interpretare scurtă
Completează în notebook, în 3–5 rânduri:

Ce două axe ai ales?

fear + anti_corruption

De ce le-ai ales?

pentru că o temă recurentă în comentariile la videoclipurile politicienilor/creatori de conținut care discută evenimente politice este neîncrederea în instituții și autorități.. împreună cu teama în general față de viitor/situația statului, etc


Modelul a returnat JSON corect?

da


Care a fost cea mai mare problemă?

uneori nu înțelege destul de bine axele .. 



Ce ai schimba în prompt?

adăugarea de exemple sau definiții mai detaliate pentru fiecare axă